# 費用報銷分析

此筆記本示範如何建立使用外掛程式的代理人，處理本地收據影像中的差旅費用，生成費用報銷電子郵件，並使用圓餅圖視覺化費用資料。代理人會根據任務上下文動態選擇函式。

步驟：
1. OCR 代理人處理本地收據影像並擷取差旅費用資料。
2. 電子郵件代理人生成費用報銷電子郵件。

### 差旅費用情境範例：
假設你是一名員工，正在另一城市出差參加商務會議。公司有政策報銷所有合理的差旅相關費用。以下是可能的差旅費用明細：
- 交通：
往返你所在城市與目的城市的機票費用。
機場接送計程車或叫車服務費用。
目的城市的本地交通費用（如公共運輸、租車或計程車）。

- 住宿：
距離會議地點近的中階商務旅館三晚住宿費用。

- 餐飲：
根據公司每日津貼政策，包含早餐、午餐和晚餐的每日餐飲補助。

- 雜項費用：
機場停車費用。
旅館網路連線費用。
小費或其他服務費。

- 文件：
你提交所有收據（機票、計程車、旅館、餐飲等）和填寫完成的費用報告以申請報銷。


## 匯入所需的函式庫

匯入此筆記本所需的函式庫和模組。


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## 定義費用模型

 建立一個用於個別費用的 Pydantic 模型，以及一個 ExpenseFormatter 類別，用來將使用者查詢轉換成結構化的費用資料。

 每筆費用將以如下格式表示：
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## 定義工具 - 產生電子郵件

建立一個用於提交報銷申請的電子郵件產生工具函式。 
- 此工具使用 Microsoft Agent Framework 的 `@tool` 裝飾器。 
- 它會計算報銷費用的總金額，並將明細格式化成電子郵件內容。


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# 從收據影像中擷取旅遊費用的工具

建立一個工具函式以從收據影像中擷取旅遊費用。
- 此工具使用 Microsoft Agent Framework 的 `@tool` 裝飾器。
- 它會讀取收據影像，並將其編碼為 base64，然後回傳資料 URI 供代理程式分析。


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## 處理費用

定義代理並使用 `WorkflowBuilder` 將它們串接成一個連續的工作流程。  
- OCR 代理使用 `load_receipt_image` 工具從收據影像中擷取結構化的費用資料。  
- 電子郵件代理接收擷取的資料，並使用 `generate_expense_email` 工具產生專業的費用報銷電子郵件。  
- 使用帶有 `add_edge` 的 `WorkflowBuilder` 建立一個連續的流程管線：OCR 代理 → 電子郵件代理。


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## 主要功能

建立順序工作流程並執行它，以處理收據圖像並生成費用申請電子郵件。


> **注意：** 此工作流程目前將收據影像作為 base64 文字傳遞，大多數聊天模型（包括 gpt-4o）不會將其視為影像。
> 它也可能超過模型的上下文視窗大小。建議使用 Azure AI Vision（或其他 OCR 工具）進行 OCR，只傳遞擷取的文字，或重構以將影像作為 `image_url` 訊息發送。
> 如果您只是想避免上下文錯誤，請嘗試使用較小的收據影像或具備較大上下文視窗的模型。


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
此文件已使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們努力追求準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於關鍵資訊，建議採用專業人工翻譯。我們不對因使用此翻譯所產生的任何誤解或誤譯承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
